# Robustness & validasi

Tier 1 wajib (horizon, lag, sub-periode 2021-04-05, tanpa COVID/Brent ekstrem, dekomposisi dolar global, bulanan), validasi V2 (studi kasus 2026) & V4 (LP vs event study); Tier 2 opsional. **Sedang–berat** — disarankan Kaggle/Colab.

Input: output notebook 00, 01 (`profiles`, `events`), dan 04 (`forecast_eval`).

Logika ada di `src/` (`kagami.pipelines.robustness`); notebook ini hanya menjalankan dan memvisualisasikan. Spesifikasi: `docs/METHODS.md`, status WP: `docs/WORKPLAN.md`.

In [ ]:
# === 0. Setup runtime: lokal / Kaggle / Colab (sel ini IDENTIK di semua notebook) ===
import glob, importlib.util, os, subprocess, sys
from pathlib import Path

DRIVE_DIR = "/content/drive/MyDrive/kagami"   # Colab: folder project di Google Drive (ubah bila beda)

def _find_code_root() -> Path:
    cands = []
    if "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG"):
        from google.colab import drive
        drive.mount("/content/drive")
        cands.append(Path(DRIVE_DIR))
    cands += [Path.cwd(), Path.cwd().parent]
    cands += [Path(p).parents[1] for p in glob.glob("/kaggle/input/**/configs/params.yaml", recursive=True)]
    for c in cands:
        if (c / "configs" / "params.yaml").exists() and (c / "src" / "__init__.py").exists():
            return c.resolve()
    raise FileNotFoundError("Kode KAGAMI tidak ditemukan. Kaggle: tambahkan dataset 'kagami-code'. "
                            "Colab: salin project ke DRIVE_DIR. Lokal: jalankan dari folder notebooks/.")

ROOT = _find_code_root()
_need = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "sklearn": "scikit-learn",
         "pyarrow": "pyarrow", "openpyxl": "openpyxl", "yaml": "pyyaml", "plotly": "plotly"}
_missing = [pip for mod, pip in _need.items() if importlib.util.find_spec(mod) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])
if importlib.util.find_spec("kagami") is None:   # paket = folder src/ (D-024)
    _spec = importlib.util.spec_from_file_location("kagami", ROOT / "src" / "__init__.py",
                                                   submodule_search_locations=[str(ROOT / "src")])
    _mod = importlib.util.module_from_spec(_spec)
    sys.modules["kagami"] = _mod
    _spec.loader.exec_module(_mod)
print("kode:", ROOT)

In [ ]:
from kagami.utils.env import artifact_status, init_runtime
from kagami.utils.io import read_artifact

cfg = init_runtime(drive_dir=DRIVE_DIR)

## Cek input

In [ ]:
from kagami.pipelines import robustness
from kagami.utils.io import require

require(cfg, *robustness.REQUIRES)
artifact_status(cfg)

## Jalankan tahap
Stub yang belum diimplementasikan berhenti dengan `NotImplementedYet: WP-xx`.

In [ ]:
from kagami.pipelines import robustness
robustness.run(cfg, include_tier2=False)
artifact_status(cfg, stage="robustness")

## Forest plot robustness

In [ ]:
from kagami.viz import plots
rob = read_artifact(cfg, "robustness")
plots.forest(rob, ["check_id", "commodity", "horizon"], "Robustness vs baseline").show()
rob

## Simpan output
- **Kaggle**: *Save Version → Save & Run All*. Output (`/kaggle/working`) lalu ditambahkan sebagai **input** di notebook
  tahap berikutnya (*Add Input → Notebook Output*); `init_runtime` menemukannya otomatis.
- **Colab**: artefak sudah tertulis ke `DRIVE_DIR` di Google Drive; notebook berikutnya membaca dari folder yang sama.
- **Lokal**: artefak tertulis ke folder project. Salin `results/` + `data/processed/` hasil cloud ke project lokal sebelum
  menjalankan app / `python -m kagami package`.